# Unified validation and held-out uplift evaluation

This notebook selects model variants with training-only out-of-fold (OOF) Qini, then evaluates each selected family once on the untouched test split. All files are fingerprint-verified against the canonical outcome and treatment arrays before scoring.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    MODEL_RESULTS_DIR,
    PREDICTIONS_DIR,
    PROCESSED_DATA_DIR,
)
from src.metrics import compute_qini_auuc, validate_prediction_frame
from src.plotting import plot_cumulative_gain_curve, plot_qini_curve, save_fig

MODEL_RESULTS_DIR.mkdir(parents=True, exist_ok=True)


## Load canonical train/test fingerprints and prediction files


In [ ]:
y_train = pd.read_csv(PROCESSED_DATA_DIR / "y_train.csv").squeeze("columns")
treatment_train = pd.read_csv(PROCESSED_DATA_DIR / "treatment_train.csv").squeeze("columns")
y_test = pd.read_csv(PROCESSED_DATA_DIR / "y_test.csv").squeeze("columns")
treatment_test = pd.read_csv(PROCESSED_DATA_DIR / "treatment_test.csv").squeeze("columns")

test_frames = {
    "response": pd.read_csv(PREDICTIONS_DIR / "response_baseline_predictions.csv"),
    "s": pd.read_csv(PREDICTIONS_DIR / "s_learner_predictions.csv"),
    "t": pd.read_csv(PREDICTIONS_DIR / "t_learner_predictions.csv"),
    "x": pd.read_csv(PREDICTIONS_DIR / "tau_hat_x.csv"),
    "r": pd.read_csv(PREDICTIONS_DIR / "tau_hat_r.csv"),
    "forest": pd.read_csv(PREDICTIONS_DIR / "causal_forest_predictions.csv"),
}
oof_frames = {
    "response": pd.read_csv(PREDICTIONS_DIR / "response_baseline_oof_predictions.csv"),
    "s": pd.read_csv(PREDICTIONS_DIR / "s_learner_oof_predictions.csv"),
    "t": pd.read_csv(PREDICTIONS_DIR / "t_learner_oof_predictions.csv"),
    "x": pd.read_csv(PREDICTIONS_DIR / "tau_hat_x_oof.csv"),
    "r": pd.read_csv(PREDICTIONS_DIR / "tau_hat_r_oof.csv"),
    "forest": pd.read_csv(PREDICTIONS_DIR / "causal_forest_oof_predictions.csv"),
}

for name, frame in test_frames.items():
    validate_prediction_frame(
        frame,
        y_test,
        treatment_test,
        frame_name=f"{name} test predictions",
        row_id_col="test_row_id",
    )
for name, frame in oof_frames.items():
    validate_prediction_frame(
        frame,
        y_train,
        treatment_train,
        frame_name=f"{name} OOF predictions",
        row_id_col="train_row_id",
    )

print("All prediction files match the canonical train/test fingerprints.")


## Score every candidate on OOF validation predictions

The response models are non-causal ranking benchmarks. Their scores show the treatment-effect signal captured by their ranking; they are not CATE estimates.


In [ ]:
validation_tau = {
    "Response Baseline (LR)": oof_frames["response"]["logistic_proba"].to_numpy(),
    "Response Baseline (XGB)": oof_frames["response"]["xgboost_proba"].to_numpy(),
    "S-Learner (LR)": oof_frames["s"]["tau_hat_lr"].to_numpy(),
    "S-Learner (XGB)": oof_frames["s"]["tau_hat_xgb"].to_numpy(),
    "T-Learner (LR)": oof_frames["t"]["lgr_predicted_uplift"].to_numpy(),
    "T-Learner (XGB)": oof_frames["t"]["xgb_predicted_uplift"].to_numpy(),
    "T-Learner (RF)": oof_frames["t"]["rf_predicted_uplift"].to_numpy(),
    "X-Learner": oof_frames["x"]["tau_hat_x"].to_numpy(),
    "R-Learner": oof_frames["r"]["tau_hat_r"].to_numpy(),
    "Uplift Random Forest": oof_frames["forest"]["cf_predicted_uplift"].to_numpy(),
}
validation_scores = compute_qini_auuc(
    y_train, treatment_train, validation_tau, normalize=True
)
validation_comparison = pd.DataFrame({
    "validation_qini": validation_scores["qini_score"],
    "validation_auuc": validation_scores["auuc_score"],
}).sort_values("validation_qini", ascending=False)
display(validation_comparison)


## Select variants without consulting test performance


In [ ]:
family_candidates = {
    "Response Baseline": ["Response Baseline (LR)", "Response Baseline (XGB)"],
    "S-Learner": ["S-Learner (LR)", "S-Learner (XGB)"],
    "T-Learner": ["T-Learner (LR)", "T-Learner (XGB)", "T-Learner (RF)"],
    "X-Learner": ["X-Learner"],
    "R-Learner": ["R-Learner"],
    "Uplift Random Forest": ["Uplift Random Forest"],
}

selected_by_family = {
    family: validation_scores["qini_score"][candidates].idxmax()
    for family, candidates in family_candidates.items()
}
selection_df = pd.DataFrame(
    [{"family": family, "selected_model": model}
     for family, model in selected_by_family.items()]
)
display(selection_df)


## Final held-out test evaluation


In [ ]:
test_tau = {
    "Response Baseline (LR)": test_frames["response"]["logistic_proba"].to_numpy(),
    "Response Baseline (XGB)": test_frames["response"]["xgboost_proba"].to_numpy(),
    "S-Learner (LR)": test_frames["s"]["tau_hat_lr"].to_numpy(),
    "S-Learner (XGB)": test_frames["s"]["tau_hat_xgb"].to_numpy(),
    "T-Learner (LR)": test_frames["t"]["lgr_predicted_uplift"].to_numpy(),
    "T-Learner (XGB)": test_frames["t"]["xgb_predicted_uplift"].to_numpy(),
    "T-Learner (RF)": test_frames["t"]["rf_predicted_uplift"].to_numpy(),
    "X-Learner": test_frames["x"]["tau_hat_x"].to_numpy(),
    "R-Learner": test_frames["r"]["tau_hat_r"].to_numpy(),
    "Uplift Random Forest": test_frames["forest"]["cf_predicted_uplift"].to_numpy(),
}

selected_test_tau = {
    selected_model: test_tau[selected_model]
    for selected_model in selected_by_family.values()
}
test_scores = compute_qini_auuc(
    y_test, treatment_test, selected_test_tau, normalize=True
)

rows = []
for family, selected_model in selected_by_family.items():
    validation_qini = float(validation_scores["qini_score"][selected_model])
    validation_auuc = float(validation_scores["auuc_score"][selected_model])
    test_qini = float(test_scores["qini_score"][selected_model])
    test_auuc = float(test_scores["auuc_score"][selected_model])
    rows.append({
        "family": family,
        "selected_model": selected_model,
        "validation_qini": validation_qini,
        "test_qini": test_qini,
        "qini_generalization_gap": validation_qini - test_qini,
        "validation_auuc": validation_auuc,
        "test_auuc": test_auuc,
        "auuc_generalization_gap": validation_auuc - test_auuc,
    })

comparison_df = pd.DataFrame(rows).sort_values("test_qini", ascending=False)
display(comparison_df)

comparison_path = MODEL_RESULTS_DIR / "validation_test_uplift_comparison.csv"
selection_path = MODEL_RESULTS_DIR / "model_variant_validation_scores.csv"
comparison_df.to_csv(comparison_path, index=False)
validation_comparison.reset_index(names="model").to_csv(selection_path, index=False)
print("Saved:", comparison_path)
print("Saved:", selection_path)


## Final test curves for validation-selected models


In [ ]:
fig_qini = plot_qini_curve(
    y_test, treatment_test, selected_test_tau, figsize=(10, 8)
)
save_fig(fig_qini, "model_comparison_qini_curves_causalml.png")

fig_gain = plot_cumulative_gain_curve(
    y_test, treatment_test, selected_test_tau, figsize=(10, 8)
)
save_fig(fig_gain, "model_comparison_auuc_curves_causalml.png")


## Interpretation

Use `validation_test_uplift_comparison.csv` as the paper's overfitting/underfitting evidence. Large positive validation-minus-test gaps indicate optimistic validation performance; similarly weak values on both splits indicate underfitting. Model selection remains based exclusively on OOF validation Qini, while the test columns are final reporting evidence.
